# The Bot Range (TBR) - Official Starter Bot

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ben-uphoff/tbr-bdk/blob/main/starter_bot.ipynb)

Welcome to **The Bot Range (TBR)**. This notebook runs an autonomous bot client against the TBR REST API.

### Mission 01 Objective
1. Authenticate with your API key.
2. Spawn and survey your bot's environment.
3. Defeat hostiles, loot resources, and navigate along the extraction vector towards Ring 0.
4. Successfully extract banked spoils!

In [ ]:
# Step 1: Install httpx if not already installed
!pip install -q httpx

In [ ]:
# Step 2: Configure your API Credentials
import os

# Set your TBR API Key here or through Colab Secrets (userdata)
TBR_API_KEY = os.environ.get("TBR_API_KEY") or "YOUR_API_KEY_HERE"
TBR_API_URL = "https://api.thebotrange.com/api/v1"

print("Ready to connect to:", TBR_API_URL)

In [ ]:
# Step 3: Run the Autonomous Bot Loop
import time
import httpx

headers = {
    "Authorization": f"Bearer {TBR_API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

def api_call(method, path, payload=None):
    url = f"{TBR_API_URL}{path}"
    try:
        with httpx.Client(timeout=10.0) as client:
            res = client.request(method, url, headers=headers, json=payload)
            match res.status_code:
                case 200 | 201 | 202:
                    return res.json()
                case 429:
                    print(f"[429 Rate Limited] {res.text}")
                    return None
                case 502:
                    print(f"[502 Signal Interference] {res.text}")
                    return None
                case _:
                    print(f"[HTTP {res.status_code}] {res.text}")
                    return None
    except Exception as e:
        print(f"Network error: {e}")
        return None

def run_step():
    state = api_call("GET", "/game/state")
    if not state:
        return

    player = state.get("player", {})
    zone = state.get("zone", {})
    tile = state.get("current_tile", {})
    limiter = state.get("action_limiter", {})

    if not player.get("spawned", False) or state.get("status") in ["inactive", "despawned"]:
        print("[Status] Bot is offline. Calling POST /game/spawn ...")
        spawn_res = api_call("POST", "/game/spawn")
        if spawn_res and spawn_res.get("status") == "queued":
            wait_secs = spawn_res.get("retry_after_seconds", 5)
            print(f"[Spawn Queued] Position {spawn_res.get('queue_position')} — retrying in {wait_secs}s ...")
            time.sleep(wait_secs)
        else:
            print(spawn_res)
        return

    vector = zone.get("extraction_vector", {})
    p_zone = player.get("zone") or zone.get("coords") or [0, 0]
    p_tile = player.get("tile") or [0, 0]

    print(f"[Tick] HP: {player.get('hp')}/{player.get('max_hp')} | Pos: Zone {p_zone[0]},{p_zone[1]} Tile {p_tile[0]},{p_tile[1]} | Tokens: {limiter.get('tokens', 0)}")

    if zone.get("is_extraction_zone"):
        print("[Extraction] In Ring 0! Extracting ...")
        print(api_call("POST", "/game/extract"))
        return

    if tile.get("enemies"):
        target = tile["enemies"][0]
        print(f"[Combat] Attacking {target.get('name')} ...")
        res = api_call("POST", "/game/attack", {"enemy_id": target.get("id"), "heavy": False})
        print(res.get("combat_log") if res else "Attack failed")
        return

    if tile.get("resources"):
        print("[Loot] Harvesting resources ...")
        print(api_call("POST", "/game/loot"))
        return

    direction = vector.get("direction") or "s"
    print(f"[Move] Advancing {direction.upper()} towards center ...")
    res = api_call("POST", "/game/move", {"direction": direction, "zone_change": True})
    if res:
        print(f"[Move Success] Zone {res.get('zone')} Tile {res.get('tile')} | Free move: {res.get('free_move')} ({res.get('free_move_reason')})")

# Run 10 ticks for demonstration
for i in range(10):
    print(f"--- Iteration {i+1} ---")
    run_step()
    time.sleep(1.0)